# Valores Ausentes (*Missing Values*)

Notebook de aprendizado baseado no `section_5_missing_values.ipynb` do curso **Machine Learning Process**.

Vamos aprender a tratar valores nulos (`NaN`) usando um dataset de **Customer Lifetime Value (CLV)** — o valor monetário total que um cliente representa para o negócio ao longo do tempo.

### O que vamos cobrir:
1. Verificar valores ausentes
2. Remover nulos (`dropna`)
3. Imputação por Média / Mediana / Moda
4. Imputação Múltipla por Regressão (`IterativeImputer`)
5. Imputação por Vizinhos Mais Próximos (`KNNImputer`)
6. Comparação entre técnicas

## 1. Importar Bibliotecas

| Biblioteca | Para que serve |
|---|---|
| `pandas` | Manipulação do DataFrame, `fillna`, `dropna` |
| `numpy` | Cálculo de média e mediana |
| `scipy.stats` | Cálculo da moda |
| `sklearn.impute` | Imputação avançada (regressão iterativa e KNN) |

> **Atenção:** `enable_iterative_imputer` precisa ser importado **antes** de `IterativeImputer` — sem ele o código quebra, pois ainda é uma feature experimental no scikit-learn.

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer, KNNImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

import warnings
warnings.filterwarnings('ignore')

print('Bibliotecas carregadas com sucesso!')

## 2. Carregar os Dados

O dataset contém informações de clientes: `age`, `gender`, `income`, `days_on_platform`, `city` e `purchases`.

Criamos `lifetime_value` como proxy do CLV:
```
lifetime_value = purchases × 20
```

Algumas colunas têm **valores ausentes propositalmente** para treinarmos as técnicas.

In [ ]:
df = pd.read_csv('../5_data_preprocessing/5_1_missing_values/clv_data.csv', index_col=0)
df['lifetime_value'] = df['purchases'] * 20

print(f'Shape do dataset: {df.shape}')
df.head()

## 3. Verificar Valores Ausentes

Antes de qualquer tratamento, sempre **audite os nulos**. Existem duas formas úteis:

### 3.1 Contagem simples

`.isnull().sum()` retorna o número absoluto de nulos por coluna — rápido para ter uma visão geral.

In [ ]:
df.isnull().sum()

### 3.2 Tabela com percentual

Percentual importa porque **5% de nulos tem tratamento diferente de 60%**. Alta porcentagem pode indicar que o dado nunca foi coletado — nesse caso, talvez valha descartar a coluna inteira.

In [ ]:
def nulls_summary_table(df):
    """
    Retorna contagem e % de nulos por coluna.
    """
    null_values = pd.DataFrame(df.isnull().sum())
    null_values[1] = null_values[0] / len(df)
    null_values.columns = ['null_count', 'null_pct']
    return null_values

nulls_summary_table(df)

## 4. Técnica 1 — Remover Nulos (`dropna`)

### Como funciona
Remove **todas as linhas** que contêm pelo menos um valor nulo. A abordagem mais simples.

### Quando usar
- Poucos nulos (< 5% das linhas)
- Os nulos são **aleatórios** (MCAR — *Missing Completely At Random*): a ausência não carrega informação sobre o alvo
- O dataset é grande o suficiente para perder linhas sem prejudicar o treino

### Quando evitar
- Muitos nulos: remover 30% das linhas desperdiça sinal
- Nulos **não-aleatórios**: se usuários de alta renda tendem a não preencher `income`, removê-los cria viés — o modelo aprende com uma amostra que não representa a população real

| Prós | Contras |
|---|---|
| Zero complexidade | Perde dados — reduz tamanho do treino |
| Sem risco de introduzir ruído | Pode criar viés se a ausência não for aleatória |
| Rápido de implementar | Problemas se nulos aparecerem em novas predições em produção |

In [ ]:
drop_df = df.copy()  # sempre copiar antes de modificar!

print(f'Linhas antes do dropna: {len(drop_df)}')
drop_df = drop_df.dropna()
print(f'Linhas depois do dropna: {len(drop_df)}')
print(f'Linhas removidas: {len(df) - len(drop_df)} ({(len(df) - len(drop_df))/len(df)*100:.1f}%)')

In [ ]:
# Separar features e alvo para o modelo de comparação final
X_d = drop_df[['age', 'days_on_platform', 'income']]
y_d = drop_df['lifetime_value']

X_train_d = X_d[:4000]
y_train_d = y_d[:4000]
X_test_d  = X_d[4000:]
y_test_d  = y_d[4000:]

print(f'Treino: {X_train_d.shape} | Teste: {X_test_d.shape}')

## 5. Técnica 2 — Imputação por Média / Mediana / Moda

Substitui o `NaN` por um **resumo estatístico** calculado nos dados de treino. Simples e interpretável.

> ⚠️ **Regra de ouro anti-leakage:** calcule a estatística **sempre no conjunto de treino** e aplique no teste. Usar o teste para calcular a média seria *data leakage* — o modelo estaria "vendo o futuro".

### 5.1 Média (*mean*)

- **Quando usar:** variáveis numéricas contínuas com distribuição **simétrica**, sem outliers fortes
- **Problema:** sensível a outliers. Se `income` tem valores extremos, a média sobe — todos os nulos viram esse valor distorcido

In [ ]:
m_df = df.copy()

X_m = m_df[['age', 'days_on_platform', 'income']]
y_m = m_df['lifetime_value']

X_train_m = X_m[:4000].copy()
y_train_m = y_m[:4000]
X_test_m  = X_m[4000:].copy()
y_test_m  = y_m[4000:]

# Calcula a média NO TREINO e aplica nos dois conjuntos
age_mean  = np.mean(X_train_m['age'])
days_mean = np.mean(X_train_m['days_on_platform'])

X_train_m['age']              = X_train_m['age'].fillna(age_mean)
X_test_m['age']               = X_test_m['age'].fillna(age_mean)   # usa média do TREINO

X_train_m['days_on_platform'] = X_train_m['days_on_platform'].fillna(days_mean)
X_test_m['days_on_platform']  = X_test_m['days_on_platform'].fillna(days_mean)

print(f'Média de age usada para imputação: {age_mean:.2f}')
print(f'Nulos restantes no treino: {X_train_m.isnull().sum().sum()}')

### 5.2 Mediana (*median*)

- **Quando usar:** variáveis numéricas com distribuição **assimétrica** ou com outliers (salários, preços, idades com extremos)
- **Por que é mais robusta:** a mediana é o valor do meio da distribuição ordenada — outliers não a movem tanto quanto a média

| Distribuição | Escolha |
|---|---|
| Simétrica, sem outliers | Média |
| Assimétrica ou com outliers | **Mediana** |

In [ ]:
med_df = df.copy()

age_median = np.median(med_df['age'].dropna())
print(f'Mediana de age: {age_median}')

med_df['age'] = med_df['age'].fillna(age_median)

print(f'Nulos em age após mediana: {med_df["age"].isnull().sum()}')

### 5.3 Moda (*mode*)

- **Quando usar:** variáveis **categóricas** (país, plano, gênero) — média e mediana não fazem sentido para texto ou classes discretas
- `stats.mode()` retorna o valor mais frequente

In [ ]:
mode_df = df.copy()

# Moda de variável categórica
gender_mode = stats.mode(mode_df['gender'].dropna(), keepdims=True)[0][0]
print(f'Moda de gender: {gender_mode}')

mode_df['gender'] = mode_df['gender'].fillna(gender_mode)

print(f'Nulos em gender após moda: {mode_df["gender"].isnull().sum()}')

### Limitação geral da média/mediana/moda

Todos os nulos de uma coluna viram **o mesmo valor fixo**. Isso:
- Comprime a variância da coluna artificialmente
- Pode esconder padrões reais por trás dos nulos

Para mais nulos ou dados com estrutura complexa → usar as técnicas avançadas abaixo.

## 6. Técnica 3 — Imputação Múltipla por Regressão (`IterativeImputer`)

### Como funciona

Em vez de um valor fixo, usa **um modelo de ML** para prever o valor ausente com base nas **outras colunas**. Itera várias vezes, refinando cada imputação. Isso é chamado de **MICE** (*Multiple Imputation by Chained Equations*) na literatura estatística.

```
Rodada 1: imputa age    → usa days_on_platform, income (preenchidos na rodada anterior)
          imputa income → usa age (já imputado), days_on_platform
Rodada 2: reimputa age  → agora income está melhor → age melhora também
...
Rodada 10: convergência
```

### Parâmetros principais

| Parâmetro | O que faz | Valor típico |
|---|---|---|
| `estimator` | Modelo usado para prever o nulo | `BayesianRidge` (default) ou `RandomForestRegressor` |
| `max_iter` | Número de rodadas de imputação | 10 |
| `random_state` | Reprodutibilidade | 0 |
| `add_indicator` | Cria coluna `0/1` indicando onde foi imputado | `True` (recomendado) |
| `missing_values` | Tipo do valor nulo a imputar | `np.nan` (default) |

### Por que `add_indicator=True` é importante?

A **ausência** de um valor pode carregar informação (ex.: usuários que não informam renda podem ter padrão de compra diferente). Com o indicador, o modelo downstream pode capturar esse padrão:

| age | age_missing_indicator |
|---|---|
| 35 | 0 |
| 28 | 1 ← era NaN, foi imputado |

In [ ]:
r_df = df.copy()

X_r = r_df[['age', 'days_on_platform', 'income']]
y_r = r_df['lifetime_value']

X_train_r = X_r[:4000].copy()
y_train_r = y_r[:4000]
X_test_r  = X_r[4000:].copy()
y_test_r  = y_r[4000:]

print(f'Nulos no treino antes da imputação:\n{X_train_r.isnull().sum()}\n')

# Criar e treinar o imputer APENAS no treino
imp = IterativeImputer(max_iter=10, random_state=0)
imp.fit(X_train_r)

# Transformar treino e teste
X_train_r_imp = pd.DataFrame(imp.transform(X_train_r), columns=X_train_r.columns)
X_test_r_imp  = pd.DataFrame(imp.transform(X_test_r),  columns=X_test_r.columns)

print(f'Nulos no treino após imputação: {X_train_r_imp.isnull().sum().sum()}')
X_train_r_imp.head()

### Estimadores disponíveis

| Estimador | Descrição | Quando usar |
|---|---|---|
| `BayesianRidge` | Regressão linear regularizada (default) | Relações lineares entre features; mais rápido |
| `RandomForestRegressor` | Captura relações não-lineares | Dados complexos; mais lento; equivalente ao `missForest` do R |

In [ ]:
from sklearn.ensemble import RandomForestRegressor as RF

# Usando Random Forest como estimador (missForest)
imp_rf = IterativeImputer(estimator=RF(n_estimators=10, random_state=0), max_iter=5, random_state=0)
imp_rf.fit(X_train_r)

X_train_rf = pd.DataFrame(imp_rf.transform(X_train_r), columns=X_train_r.columns)
print('Imputação com RandomForest concluída!')
print(f'Nulos restantes: {X_train_rf.isnull().sum().sum()}')

## 7. Técnica 4 — Imputação por Vizinhos Mais Próximos (`KNNImputer`)

### Como funciona

Para cada valor ausente, encontra os **K registros mais similares** (vizinhos mais próximos) que têm valor naquele campo — e usa a média (ou média ponderada) desses vizinhos como imputação.

```
Registro com age = NaN, income = 50.000, days = 200
→ Encontra os 5 registros mais parecidos (por income e days)
→ age desses 5 vizinhos: [28, 31, 29, 33, 30]
→ Imputa age = média(28, 31, 29, 33, 30) = 30.2
```

### Parâmetros principais

| Parâmetro | O que faz | Valor típico |
|---|---|---|
| `n_neighbors` | Número de vizinhos usados | 5 (default) |
| `weights` | Como ponderar os vizinhos | `"uniform"` (igual) ou `"distance"` (mais perto = mais peso) |
| `metric` | Métrica de distância | `"nan_euclidean"` (default — ignora NaN no cálculo) |
| `add_indicator` | Coluna indicadora de imputação | `False` (default) |

### `weights` — uniform vs. distance

| `weights` | Comportamento |
|---|---|
| `"uniform"` | Todos os K vizinhos têm peso igual — média simples |
| `"distance"` | Vizinhos mais próximos têm mais influência — média ponderada pelo inverso da distância |

### Cuidado com escala!

Distância Euclidiana é sensível à escala. Se `income` está na casa de milhares e `age` está em dezenas, `income` vai dominar o cálculo de distância. Recomenda-se escalar as features antes.

In [ ]:
# Usar os dados já preparados do passo anterior (sem nulos no treino)
X_train_for_knn = X_r[:4000].copy()
X_test_for_knn  = X_r[4000:].copy()

imputer_knn = KNNImputer(n_neighbors=5, weights="uniform")
imputer_knn.fit(X_train_for_knn)

X_train_k = pd.DataFrame(imputer_knn.transform(X_train_for_knn), columns=X_train_for_knn.columns)
X_test_k  = pd.DataFrame(imputer_knn.transform(X_test_for_knn),  columns=X_test_for_knn.columns)

y_train_k = y_r[:4000].reset_index(drop=True)
y_test_k  = y_r[4000:].reset_index(drop=True)

print(f'Nulos após KNN: {X_train_k.isnull().sum().sum()}')
X_train_k.head()

### Escolha de K — overfitting vs. underfitting

| K pequeno (ex: 1–3) | K grande (ex: 20+) |
|---|---|
| Imputação mais específica | Imputação mais suavizada |
| Risco de overfitting (depende muito de poucos pontos) | Perde precisão local |

**K=5** é um bom ponto de partida.

In [ ]:
# Comparar K=3 vs K=5 vs K=10
for k in [3, 5, 10]:
    imp_k = KNNImputer(n_neighbors=k)
    imp_k.fit(X_train_for_knn)
    X_tr = pd.DataFrame(imp_k.transform(X_train_for_knn), columns=X_train_for_knn.columns)
    print(f'K={k} | Média de age imputado: {X_tr["age"].mean():.2f} | Desvio: {X_tr["age"].std():.2f}')

## 8. Comparação entre Técnicas

Treinamos um `RandomForestRegressor` com cada conjunto de dados imputado e comparamos o **MAE** (*Mean Absolute Error* — erro médio absoluto) nas predições de `lifetime_value`.

**MAE** = em média, o quanto a predição erra em unidades do alvo. **Menor = melhor.**

In [ ]:
results = {}

# ── 1. Drop Null ────────────────────────────────────────────────
clf = RandomForestRegressor(random_state=0, n_estimators=50)
clf.fit(X_train_d, y_train_d)
results['Drop Null'] = mean_absolute_error(y_test_d, clf.predict(X_test_d))

# ── 2. Média ─────────────────────────────────────────────────────
clf2 = RandomForestRegressor(random_state=0, n_estimators=50)
clf2.fit(X_train_m, y_train_m)
results['Média'] = mean_absolute_error(y_test_m, clf2.predict(X_test_m))

# ── 3. IterativeImputer ──────────────────────────────────────────
clf3 = RandomForestRegressor(random_state=0, n_estimators=50)
clf3.fit(X_train_r_imp, y_train_r)
results['Regressão Iterativa'] = mean_absolute_error(y_test_r, clf3.predict(X_test_r_imp))

# ── 4. KNNImputer ────────────────────────────────────────────────
clf4 = RandomForestRegressor(random_state=0, n_estimators=50)
clf4.fit(X_train_k, y_train_k)
results['KNN'] = mean_absolute_error(y_test_k, clf4.predict(X_test_k))

# ── Resultado ────────────────────────────────────────────────────
print('=' * 40)
print(f'{"Técnica":<25} {"MAE":>10}')
print('=' * 40)
for nome, mae in sorted(results.items(), key=lambda x: x[1]):
    print(f'{nome:<25} {mae:>10.3f}')
print('=' * 40)
print('(menor MAE = melhor)')

## 9. Resumo Comparativo

| Técnica | Complexidade | Quando usar | Risco principal |
|---|---|---|---|
| **Drop Null** | Muito baixa | Poucos nulos aleatórios | Perde dados, pode criar viés |
| **Média** | Baixa | Distribuição simétrica, pouco tempo | Sensível a outliers, comprime variância |
| **Mediana** | Baixa | Distribuição assimétrica ou outliers | Comprime variância |
| **Moda** | Baixa | Variáveis categóricas | Comprime variância |
| **IterativeImputer** | Alta | Relações entre features importam | Mais lento, possível overfitting |
| **KNNImputer** | Média-alta | Similaridade entre registros é informativa | Lento em datasets grandes, sensível à escala |

---

## Boas Práticas

1. **Sempre use o treino** para calcular estatísticas (média, mediana, vizinhos) e aplique no teste — nunca o contrário (data leakage)
2. **`df.copy()`** antes de imputar — preserve o DataFrame original
3. **`add_indicator=True`** quando suspeitar que o padrão de ausência é informativo
4. **Escale as features** antes do KNNImputer — distância Euclidiana é sensível à escala
5. **Compare no downstream** — a melhor técnica é aquela que produz melhor MAE/AUC no modelo final, não a mais sofisticada em teoria
6. **Documente** qual técnica foi usada — em produção, o pipeline precisa repetir exatamente a mesma imputação do treino

---

## Suas notas

- 
- 